To use the AENets model please visit and follow the instructions in the original repository

https://github.com/ZhangYuanhan-AI/CelebA-Spoof/tree/master/intra_dataset_code

# Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
!cp -r "/content/gdrive/MyDrive/BiometricsModels/AENet/assets/" .

In [ ]:
import time
import sys
import tqdm
import logging

import cv2
import numpy as np

In [ ]:
import glob
import pandas as pd

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
from assets.client import get_image, verify_output
from assets.tsn_predict import TSNPredictor as CelebASpoofDetector

# Load Dataset

In [ ]:
!cp "/content/gdrive/MyDrive/DATABASES/iqa_spoof_dataset/fas_iqa_dataset.zip" .
!unzip -qq fas_iqa_dataset.zip

In [ ]:
!rm -r fas_iqa_dataset.zip

## Prepare Data to Predict

In [ ]:
np.random.seed(42)
df = pd.DataFrame([
    {
        'filename': x.split('/')[-1],
        'scene': x.split('/')[-2],
        'label': x.split('/')[-3],
        'data_type': x.split('/')[-4],
        'distortion_type': x.split('/')[-5],
        'label_iqa': x.split('/')[-1].split('_f_')[0] if len(x.split('/')[-1].split('_f_'))>1 else 'original',
        'full_path_iqa': x,
    } for x in glob.glob('fas_iqa_dataset/*/*/*/*/*')
])
print('Size DF:', df.shape[0])
df.sample(3)

Size DF: 100000


,filename,scene,label,data_type,distortion_type,label_iqa,full_path_iqa
75721,BlurY_7_f_270542.jpg,client002268_Env1_Ilum1_Spt3,attack,train,BlurY,BlurY_7,fas_iqa_dataset/BlurY/train/attack/client00226...
80184,hbright_10_f_frame_182.jpg,real_client012_laptop_SD_scene01,real,train,hbright,hbright_10,fas_iqa_dataset/hbright/train/real/real_client...
19864,noise_25_f_frame_153.jpg,attack_client008_android_SD_ipad_video_scene01,attack,train,noise,noise_25,fas_iqa_dataset/noise/train/attack/attack_clie...


## Prepare generator data

In [ ]:
column_path = 'full_path_iqa'

def batch_image_generator(df, batch_size=16, process_image_func=None):

    total_images = len(df)
    num_batches = int(np.ceil(total_images/BATCH))

    for batch_idx in range(num_batches):

        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, total_images)


        img_batch_list = []
        for idx in range(start_idx, end_idx):
            image_path = df.iloc[idx][column_path]

            try:

                img = cv2.imread(image_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))

                img_batch_list.append(img)

            except Exception as e:
                print(f"Unexpected error: {image_path}: {e}")

        yield np.array(img_batch_list, dtype=object).astype('uint8')

In [ ]:
BATCH = 64
df_process = df.copy()

image_generator_test = batch_image_generator(df_process, batch_size=BATCH)

total_images = len(df_process)
num_batches = (total_images + BATCH - 1) // BATCH
total_images, num_batches

(100000, 1563)

# Pretrained Model

## Load model

In [ ]:
detector_pretreined = CelebASpoofDetector()

/content/assets/tsn_predict.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./assets/ckpt_iter.pth.tar')


## Predict

In [ ]:
predicts = []
for batch in tqdm.tqdm(image_generator_test, total=num_batches):
    try:
        predicts.extend(detector_pretreined.predict(batch))
    except Exception as e:
        print(f"Unexpected error: {e}")
        predicts.extend([None] * len(batch))
len(predicts)

100%|██████████| 1563/1563 [15:36<00:00,  1.67it/s]


100000

In [ ]:
df_process['label_predict_proba'] = np.array(predicts).tolist()

In [ ]:
df_process.head(3)

,filename,scene,label,data_type,distortion_type,label_iqa,full_path_iqa,label_predict_proba
0,jpgcompression_30_f_523234.jpg,client005536_Env0_Ilum0_Spt0,real,test,jpgcompression,jpgcompression_30,fas_iqa_dataset/jpgcompression/test/real/clien...,"[0.9999992847442627, 7.286938625838957e-07]"
1,jpgcompression_10_f_523234.jpg,client005536_Env0_Ilum0_Spt0,real,test,jpgcompression,jpgcompression_10,fas_iqa_dataset/jpgcompression/test/real/clien...,"[0.9999756813049316, 2.4349234081455506e-05]"
2,jpgcompression_50_f_523234.jpg,client005536_Env0_Ilum0_Spt0,real,test,jpgcompression,jpgcompression_50,fas_iqa_dataset/jpgcompression/test/real/clien...,"[0.9999984502792358, 1.5407807723022415e-06]"


In [ ]:
path_dir = '/content/gdrive/MyDrive/csv_results/'

In [ ]:
df_process.to_csv(f'{path_dir}AENet_protIQA_trCeAS_tsCeAS.csv', index=False)

END